# 03. Disclosure Text — 공시 텍스트 분석

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eddmpython/dartlab/blob/master/notebooks/tutorials/03_disclosure.ipynb)

숫자 뒤에 숨겨진 텍스트를 읽는다. 모든 데이터는 property로 접근한다.

**학습 내용:**
- 사업의 내용 (섹션별 분류, 변경 탐지)
- 경영진단 및 분석의견 (MD&A)
- 회사의 개요 (설립일, 신용등급)
- 회사 기본정보 (설립일, 상장일, 대표이사)
- 원재료·설비투자 현황
- 회사 연혁

In [ ]:
# Colab 환경에서 dependency conflict 경고가 뜰 수 있으나, 무시해도 됩니다.
# (google-colab, bigframes 등 Colab 기본 패키지와의 버전 차이이며 동작에 영향 없음)
!pip install -q dartlab

In [ ]:
from dartlab import Company

samsung = Company("005930")

## 1. 사업의 내용

사업보고서의 '사업의 내용' 섹션을 구조적으로 분류한다. property는 섹션 리스트를 반환한다.

In [ ]:
sections = samsung.business

for section in sections:
    print(f"[{section.key}] {section.title} ({section.chars:,}자)")

In [ ]:
# 개요 섹션 텍스트 미리보기
overview_section = next((s for s in sections if s.key == "overview"), None)
if overview_section:
    print(overview_section.text[:500])

### 연도별 변경 탐지

텍스트 변경량을 연도별로 추적한다. `get()`으로 전체 Result를 받아야 접근 가능하다.

In [ ]:
biz = samsung.get("business")
for change in biz.changes:
    print(f"{change.year}: 변경 {change.changedPct:.1%} | +{change.added:,}자 -{change.removed:,}자 | 총 {change.totalChars:,}자")

## 2. 경영진단 및 분석의견 (MD&A)

property는 개요 텍스트를 반환한다. `get()`으로 전체 섹션에 접근한다.

In [ ]:
# property — 개요 텍스트
samsung.mdna

In [ ]:
# get()으로 전체 섹션 구조
mdna = samsung.get("mdna")
for section in mdna.sections:
    print(f"[{section.category}] {section.title} — 텍스트 {section.textLines}줄, 테이블 {section.tableLines}줄")

In [ ]:
# 사업 개요 미리보기
if mdna.overview:
    print(mdna.overview[:500])

## 3. 회사의 개요

설립일, 주소, 신용등급 등 정량 데이터를 추출한다.

In [ ]:
ov = samsung.overview

if ov:
    print(f"설립: {ov.founded}")
    print(f"소재지: {ov.address}")
    print(f"홈페이지: {ov.homepage}")
    print(f"종속회사: {ov.subsidiaryCount}개")
    print(f"중소기업: {ov.isSME}")
    print(f"상장일: {ov.listedDate}")

In [ ]:
# 신용등급
if ov:
    for cr in ov.creditRatings:
        print(f"{cr.agency}: {cr.grade}")

In [ ]:
# 파싱 상태
if ov:
    print(f"원문에 없음: {ov.missing}")
    print(f"파싱 실패: {ov.failed}")

## 4. 원재료·설비투자

In [ ]:
rm = samsung.rawMaterial

if rm:
    print("--- 원재료 매입 ---")
    for m in rm.materials[:5]:
        print(f"  {m.item}: {m.amount}백만원 ({m.ratio}%)")

In [ ]:
if rm and rm.equipment:
    eq = rm.equipment
    print(f"유형자산 합계: {eq.total}")
    print(f"CAPEX: {eq.capex}")

In [ ]:
if rm:
    for item in rm.capexItems:
        print(f"{item.segment}: {item.amount}백만원")

## 5. 회사 기본정보

설립일, 상장일, 대표이사, 본점소재지 등을 dict로 반환한다.

In [ ]:
info = samsung.companyOverviewDetail

if info:
    for k, v in info.items():
        print(f"{k}: {v}")

## 6. 회사 연혁

주요 연혁 이벤트를 날짜별로 정리한 DataFrame이다.

In [ ]:
samsung.companyHistory